# PICKO Research · NB1 — **Breadth**: how many tools before it breaks?

Finetune a **separate model per tool-set size** (nested), offer tools in **compact** form
(name + description, no parameters — so more names fit the encoder), and measure **tool selection only**.
The 1024-token encoder truncates the offered list, so past ~20 tools some are never seen — that ceiling
is the result, annotated with `n_visible`.

*Run & forget:* each size trains once, its checkpoint + the running results land in Drive, and re-running
after a restart **skips finished sizes**. Watch progress in the cell output or in `picko_out/run.log`.

## 0 · Colab quick-start (GPU) — run & forget, restart-safe

**On Colab first: Runtime → Change runtime type → GPU (L4 recommended; T4/A100 also fine).**
This cell clones the repo, pins the exact JAX/Flax, mounts Drive, and points **both** the data (in) and
the checkpoints+results (out) at your **`MyDrive/picko/`** folder — so a runtime restart loses nothing.

**Prerequisite (one-time):** `picko_balanced.jsonl` must be in `MyDrive/picko/`. **Running locally?** This
cell is a no-op — skip to cell 1.

In [ ]:
# --- Colab bootstrap (safe to re-run; no-op locally) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/picko"):
        !git clone -b hadar-work https://github.com/HadarBit/picko.git /content/picko
    %pip install -q "jax[cuda12]==0.10.2" "jaxlib==0.10.2" "flax==0.12.8"
    sys.path.insert(0, "/content/picko")
    from google.colab import drive; drive.mount("/content/drive")
    import shutil
    DRIVE = "/content/drive/MyDrive/picko"                      # <- everything lives here
    os.environ["PICKO_OUT_DIR"] = f"{DRIVE}/picko_out"          # checkpoints + results (durable)
    os.environ["PICKO_LOG"]     = f"{DRIVE}/picko_out/run.log"  # durable log across restarts
    os.makedirs(os.environ["PICKO_OUT_DIR"], exist_ok=True)
    dst = "/content/picko/data/picko_balanced.jsonl"
    if not os.path.exists(dst):
        cands = [f"{DRIVE}/picko_balanced.jsonl", "/content/drive/MyDrive/picko_balanced.jsonl"]
        src = next((c for c in cands if os.path.exists(c)), None)
        if src is None:
            have = os.listdir(DRIVE) if os.path.isdir(DRIVE) else "(MyDrive/picko not found)"
            raise FileNotFoundError(
                "picko_balanced.jsonl not found. Upload it to MyDrive/picko/. "
                f"Currently in {DRIVE}: {have}")
        os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(src, dst)
        print("copied data from", src)
    import jax
    print("GPU:");
    !nvidia-smi -L
    print("jax devices:", jax.devices())
    _plat = jax.devices()[0].platform
    assert _plat == "gpu", (
        f"JAX is running on '{_plat}', NOT the GPU — every finetune/eval will be ~30x slower "
        "(hours instead of minutes). FIX: Runtime > Change runtime type > GPU (L4), then "
        "Runtime > Restart session, and re-run this cell. If a GPU IS selected but this still "
        "fails, the CUDA plugin didn't load — re-run the %pip line above, then restart.")
    print("bootstrap OK · GPU active · data =", dst, "· OUT_DIR =", os.environ["PICKO_OUT_DIR"])
else:
    print("Not on Colab — running locally (CPU).")

## 1 · Setup & data overview

In [ ]:
# ensure the repo root is importable (works from notebooks/research/, Colab, etc.)
import os, sys
_here = os.path.abspath(os.getcwd())
for _ in range(6):
    if os.path.exists(os.path.join(_here, "scripts", "picko_research.py")): break
    _here = os.path.dirname(_here)
if os.path.isdir("/content/picko"): _here = "/content/picko"
if _here not in sys.path: sys.path.insert(0, _here)

from scripts.picko_research import *
import json, time
import pandas as pd, numpy as np, matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_theme(style="whitegrid")
except Exception:
    sns = None
from tqdm.auto import tqdm

cat, tok, raw, FOCUS, OUT_DIR = load_context()
env_report(OUT_DIR)   # jax devices + is OUT_DIR durable (Drive)?

### The 40 focus tools\nOne row per tool, with its family, category and **parameter count / bucket**.

In [ ]:
display(tools_dataframe(cat, FOCUS))

### All examples for these 40 tools\nOne row per training example (query → gold tool), tagged with the gold tool's **param bucket**.

In [ ]:
ex_df = examples_dataframe(cat, raw, FOCUS)
print("examples:", ex_df.shape[0], "| per param bucket:", ex_df["param_bucket"].value_counts().to_dict())
display(ex_df.head(10))

## 2 · Configure the sweep\nEdit `BREADTH_SIZES` to change the tool counts tested. Sizes ≤ 40 stay inside the focus; larger sizes pull extra tools from the full 75-catalog.

In [ ]:
BREADTH_SIZES  = [3, 5, 10, 20, 30, 40]   # <- edit me
CAP_PER_TOOL   = 40      # examples/tool per finetune (raise to 120 for higher fidelity)
EPOCHS         = 1
EVAL_SUBSAMPLE = 30      # cap test examples per run for faster eval; None = full
MAX_GEN_LEN    = 64      # short decode: we only score the tool NAME (salvaged by regex if JSON truncates)
BATCH_SIZE     = 8       # finetune batch. 8 is safe on L4 (kernel + train subprocess share the GPU); raise to 16 if you have headroom, lower to 4 on OOM
RUN_TRAIN      = True
FORCE_RETRAIN  = False   # True = retrain even if a checkpoint exists

pool = breadth_pool(cat, FOCUS, seed=0)
SETS = size_sets(pool, BREADTH_SIZES)
print({k: len(v) for k, v in SETS.items()})

## 3 · Finetune per size & evaluate selection\n*Resumable:* finished sizes (checkpoint + result present) are skipped. Results persist to `OUT_DIR/breadth_results.json` after **every** size.

In [ ]:
RES = os.path.join(OUT_DIR, "breadth_results.json")
done = {}
if os.path.exists(RES) and not FORCE_RETRAIN:
    for r in json.load(open(RES)): done[r["k"]] = r
    log(f"loaded {len(done)} finished size(s) from {RES}")

t_all = time.time()
for k in BREADTH_SIZES:
    ckpt = os.path.join(OUT_DIR, f"picko_breadth_k{k}_best.pkl")
    if (k in done) and os.path.exists(ckpt) and not FORCE_RETRAIN:
        log(f"k={k}: skip (already done) — selection={done[k]['selection_acc']:.3f}")
        continue
    try:
        log(f"=== start k={k} ({len(SETS[k])} tools) ===")
        R = finetune_and_eval(cat, raw, tok, SETS[k], f"breadth_k{k}", OUT_DIR,
                              cap=CAP_PER_TOOL, epochs=EPOCHS, compact=True, offer_all=k,
                              eval_subsample=EVAL_SUBSAMPLE, run_train=RUN_TRAIN,
                              force_retrain=FORCE_RETRAIN, max_gen_len=MAX_GEN_LEN,
                              batch_size=BATCH_SIZE)
        vis = int(np.median([n_visible(e["query"], json.loads(e["tools"]), tok) for e in R["test"]]))
        done[k] = {"k": k, "selection_acc": R["metrics"]["selection_acc"],
                   "name_f1": R["metrics"]["name_f1"], "parse_rate": R["metrics"]["parse_rate"],
                   "n_visible": vis, "n_test": len(R["test"])}
        json.dump([done[x] for x in sorted(done)], open(RES, "w"), indent=2)  # persist each step
        log(f"=== done k={k}: selection={done[k]['selection_acc']:.3f} visible={vis}/{k} ===")
    except Exception as e:
        log(f"k={k}: FAILED ({type(e).__name__}: {e}) — skipping; re-run to resume this size")

log(f"ALL SIZES DONE in {time.time()-t_all:.0f}s · results={RES}")
breadth = pd.DataFrame([done[x] for x in sorted(done)])
display(breadth)

## 4 · The Breadth curve

In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
ax.plot(breadth["k"], breadth["selection_acc"], "o-", color="#4C72B0", label="selection_acc")
ax.plot(breadth["k"], breadth["name_f1"], "s--", color="#55A868", label="name_f1")
wall = breadth[breadth["n_visible"] < breadth["k"]]
if len(wall):
    kw = int(wall["k"].iloc[0]); vw = int(wall["n_visible"].iloc[0])
    ax.axvline(kw, color="#C44E52", ls=":", lw=1.5)
    ax.text(kw, 0.06, f" truncation wall\n (~{vw} of {kw} tools visible)", color="#C44E52", fontsize=9, va="bottom")
ax.set_xlabel("# tools trained / offered (k)"); ax.set_ylabel("tool-selection accuracy")
ax.set_ylim(0,1.02); ax.set_title("Breadth: selection accuracy vs tool-set size"); ax.legend()
plt.tight_layout(); save_fig("breadth_curve"); plt.show()

## 5 · Read-out

- Selection holds up to ~`k` tools then drops; the red line marks where the **compact** offered list stops
  fitting the 1024-token encoder (so the extra tools are truncated away and can't be picked).
- **Takeaway:** one PICKO instance is bounded by the *context window*, not raw capacity — beyond the wall,
  a large tool set should be sharded across categorical instances.